In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        # "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    
    # Iterate through the content blocks and return the text block
    for block in message.content:
        if block.type == "text":
            return block.text
            
    return ""

In [4]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)

    # Remove the stop_sequences argument so the model can generate the full output
    text = chat(messages)

    # Clean the markdown formatting backticks from the text before parsing
    clean_text = text.replace("```json", "").replace("```", "").strip()

    return json.loads(clean_text)

In [5]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [6]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [7]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [8]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Here's a function that uses a boto3 paginator to iterate over all objects, so it works correctly for buckets with more than 1,000 objects (S3's `ListObjectsV2` returns at most 1,000 keys per call):\n\n```python\nimport boto3\n\n\ndef find_large_log_files(s3_client, bucket_name):\n    \"\"\"\n    Return the keys of all objects in `bucket_name` whose keys end with '.log'\n    and whose size is greater than 1 MB (1,048,576 bytes).\n\n    Uses pagination so it works for buckets with any number of objects.\n\n    Args:\n        s3_client: A boto3 S3 client (e.g., boto3.client(\"s3\")).\n        bucket_name: Name of the S3 bucket to scan.\n\n    Returns:\n        list[str]: Keys of matching objects.\n    \"\"\"\n    ONE_MB = 1024 * 1024\n    matching_keys = []\n\n    paginator = s3_client.get_paginator(\"list_objects_v2\")\n    for page in paginator.paginate(Bucket=bucket_name):\n        # 'Contents' is absent when a page has no objects (e.g., empty bucket)\n        for 